In [2]:
%pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
df=pd.read_csv('D:\\DA\\Huawei_Health_Analysis\\data\\processed\\health_daily_summary_cleaned.csv')

#检查缺失值情况
df.isnull().sum()

日期        0
REM时长     0
午睡时长      0
浅睡时长      0
深睡时长      0
清醒时长      0
夜间总睡眠     0
在床时间      0
睡眠效率      0
深睡比例      0
浅睡比例      0
快速眼动比例    0
是否午睡      0
星期        0
总步数       0
总时长       0
基础代谢      0
活动消耗      0
全天总消耗     0
平均心率      0
最小心率      0
最大心率      0
记录条数      0
dtype: int64

In [4]:
#合格睡眠率
normal=len(df[(df['夜间总睡眠']>7)&(df['夜间总睡眠']<9)&(df['深睡比例']>20)&(df['浅睡比例']<60)])
total=len(df)
normal_sleep_rate=round(normal/total,2)
print("合格睡眠率为：",normal_sleep_rate)

conscious=round(df['清醒时长'].mean()/(df['夜间总睡眠'].mean()*60),2)
print(f"平均清醒率为：{conscious}")

#基于我的个人数据，清醒占比正常但合格的睡眠占比偏低，可能和晚睡有关。

合格睡眠率为： 0.4
平均清醒率为：0.06


In [11]:
#午睡对夜间睡眠效率的影响
df['当晚睡眠效率']=df['睡眠效率'].shift(-1)
df['睡眠效率相对偏离']=round((df['当晚睡眠效率']-平均睡眠效率)/平均睡眠效率,2)
df[['日期','午睡时长','当晚睡眠效率','睡眠效率相对偏离']].sort_values(by='午睡时长',ascending=False).head(20)

#结论：基于我的个人数据，午睡过长几乎不会影响当晚的睡眠效率

,日期,午睡时长,当晚睡眠效率,睡眠效率相对偏离
115,2025-09-10,238.0,96.66,0.03
250,2026-01-23,215.0,95.91,0.02
126,2025-09-21,193.0,97.80,0.04
48,2025-07-05,173.0,97.68,0.04
207,2025-12-11,166.0,93.64,-0.01
167,2025-11-01,166.0,92.84,-0.01
90,2025-08-16,141.0,96.22,0.02
279,2026-02-21,132.0,94.77,0.01
108,2025-09-03,132.0,94.94,0.01
119,2025-09-14,128.0,96.13,0.02


In [9]:
#长有氧运动对当晚清醒和睡眠效率的影响
平均清醒时长=df['清醒时长'].mean()
df['当晚睡眠效率']=df['睡眠效率'].shift(-1)
df['当晚清醒时长']=df['清醒时长'].shift(-1)
df['清醒时长相对偏离']=round((df['当晚清醒时长']-平均清醒时长)/平均清醒时长,2)
df[['日期','活动消耗','当晚清醒时长','清醒时长相对偏离','当晚睡眠效率','睡眠效率相对偏离']].sort_values(by='活动消耗',ascending=False).head(20)

#结论：基于个人数据，长有氧运动对缩短当晚清醒时长有促进作用，对当晚睡眠效率有利无害。

,日期,活动消耗,当晚清醒时长,清醒时长相对偏离,当晚睡眠效率,睡眠效率相对偏离
85,2025-08-11,3913.44,12.0,-0.55,97.36,0.03
43,2025-06-30,3563.61,14.0,-0.48,97.82,0.04
15,2025-06-02,3450.64,22.0,-0.18,94.87,0.01
294,2026-03-08,3374.54,15.0,-0.44,97.06,0.03
100,2025-08-26,2969.74,15.0,-0.44,96.73,0.03
111,2025-09-06,2908.34,58.0,1.15,88.35,-0.06
56,2025-07-13,2763.05,3.0,-0.89,99.36,0.06
73,2025-07-30,2397.85,12.0,-0.55,97.30,0.03
13,2025-05-31,1802.40,22.0,-0.18,94.87,0.01
89,2025-08-15,1778.23,24.0,-0.11,94.58,0.00


In [ ]:
#按星期分组，观察不同星期对深睡比例的影响
round(df.groupby('星期').agg({'夜间总睡眠':'mean','深睡比例':'mean'}).sort_values(by='深睡比例',ascending=False).rename(columns={'夜间总睡眠':'均睡时长','深睡比例':'均深睡比例'}),2)

#结论：基于个人数据，一周内各天的深睡比例差异不大，可能和生活作息规律有关。

,均睡时长,均深睡比例
星期,,
周五,7.01,26.23
周日,7.43,25.84
周六,7.03,25.31
周一,7.86,25.07
周二,7.15,24.77
周四,6.99,24.76
周三,7.45,24.69


In [ ]:
#探讨第二天是否午睡和昨天睡眠时长的关系
df.groupby('是否午睡')['夜间总睡眠'].mean()

#结论：昨天的睡眠时长平均减少半小时，第二天午睡的可能性增加，可能和睡眠不足有关。


是否午睡
False    7.639859
True     7.171208
Name: 夜间总睡眠, dtype: float64